# 공공 Web API 실습 (버그 수정본)

**수정 내역**
- [BUG-01] `output_type : 'json'` → `output_type = 'json'` (대입 연산자 수정)
- [BUG-02] SKY 카테고리 인덱스 오류 수정 (`int(obsrValue) - 1`)
- [BUG-03] 시간 로직 주석 수정 + 제로패딩 버그 수정 (`f"{hour:02d}00"`)
- [BUG-04] API_KEY 셀 중복 제거 → 상단 1회만 정의
- [BUG-05] RN1·UUU·VEC 카테고리 출력 추가

## 공통 설정

In [ ]:
import requests
import json
import datetime
import xmltodict
from pytz import timezone

# API Key (공통) — .env 파일 사용 권장
API_KEY_encoded = os.environ.get('PUBLIC_DATA_API_KEY', '')  # .env 파일에 키를 저장하세요
API_KEY = requests.utils.unquote(API_KEY_encoded)
print('API Key 로드 완료')

## 1. 우체국 주소 검색 API (XML 응답)

In [ ]:
url = 'http://openapi.epost.go.kr/postal/retrieveNewAdressAreaCdService/retrieveNewAdressAreaCdService/getNewAddressListAreaCd'

search_Se = 'road'       # 'road' = 도로명, 'jibun' = 지번
srch_wrd  = '둔산대로 135'

parameter = {'ServiceKey': API_KEY, 'searchSe': search_Se, 'srchwrd': srch_wrd}
result = requests.get(url, params=parameter)
print('응답 코드:', result.status_code)
xml_data = result.text
xml_data

In [ ]:
dict_data = xmltodict.parse(xml_data)

header = dict_data['NewAddressListResponse']['cmmMsgHeader']
print('성공 여부:', header['successYN'])
print('전체 건수:', header['totalCount'])

raw = dict_data['NewAddressListResponse']['newAddressListAreaCd']
# 결과가 1건이면 dict, 복수건이면 list → 통일
if isinstance(raw, dict):
    raw = [raw]

print(f'\n[검색어] {srch_wrd}')
for item in raw:
    print('-' * 40)
    print(f'  우편번호   : {item["zipNo"]}')
    print(f'  도로명주소 : {item["lnmAdres"]}')
    print(f'  지번주소   : {item["rnAdres"]}')

## 2. 기상청 초단기실황 API (JSON 응답)

In [ ]:
local_time = datetime.datetime.now(timezone('Asia/Seoul'))
print('현재 로컬 시간:', local_time)

base_date = local_time.strftime('%Y%m%d')

# [BUG-03 수정] 30분 이전이면 이전 시간, 30분 이후면 현재 시간 사용
if local_time.minute < 30:
    base_hour = local_time.hour - 1
else:
    base_hour = local_time.hour

base_time = f'{base_hour:02d}00'  # 제로패딩 적용
print(f'base_date={base_date}, base_time={base_time}')

In [ ]:
url = 'http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getUltraSrtNcst'

nx_val = 67   # 예보 지점 X 좌표 (대전 서구)
ny_val = 100  # 예보 지점 Y 좌표

parameter = {
    'ServiceKey': API_KEY, 'pageNo': 1, 'numOfRows': 10, 'dataType': 'JSON',
    'base_date': base_date, 'base_time': base_time, 'nx': nx_val, 'ny': ny_val
}
r = requests.get(url, params=parameter)
print('응답 코드:', r.status_code)
dict_data = r.json()
dict_data

In [ ]:
# [BUG-05 수정] 전체 카테고리 처리
weather_items = dict_data['response']['body']['items']['item']

sky_cond  = {1: '맑음', 3: '구름 많음', 4: '흐림'}
rain_type = {0: '없음', 1: '비', 2: '비/눈', 3: '눈', 5: '빗방울'}

CATEGORY_LABEL = {
    'T1H': '기온(°C)',
    'REH': '습도(%)',
    'RN1': '1시간강수량(mm)',
    'PTY': '강수형태',
    'SKY': '하늘상태',
    'UUU': '동서바람(m/s)',
    'VVV': '남북바람(m/s)',
    'VEC': '풍향(°)',
    'WSD': '풍속(m/s)',
}

print(f'[발표 날짜] {weather_items[0]["baseDate"]}')
print(f'[발표 시간] {weather_items[0]["baseTime"]}')
print()
for item in weather_items:
    cat = item['category']
    val = item['obsrValue']
    label = CATEGORY_LABEL.get(cat, cat)

    # [BUG-02 수정] SKY: int 변환 후 딕셔너리 조회
    if cat == 'SKY':
        val = sky_cond.get(int(val), val)
    elif cat == 'PTY':
        val = rain_type.get(int(val), val)

    print(f'  {label}: {val}')

## 3. 에어코리아 API — TM 좌표 + 근접 측정소 (JSON 응답)

In [ ]:
url = 'http://apis.data.go.kr/B552584/MsrstnInfoInqireSvc/getTMStdrCrdnt'

umd_name    = '봉명동'
output_type = 'json'  # [BUG-01 수정] ':' → '=' 으로 변경

parameter = {
    'serviceKey': API_KEY, 'returnType': output_type,
    'numOfRows': 10, 'pageNo': 1, 'umdName': umd_name
}
dict_data = requests.get(url, params=parameter).json()

total = dict_data['response']['body']['totalCount']
items = dict_data['response']['body']['items']

print(f'[검색어] {umd_name}  |  검색 결과: {total}건')
for k, item in enumerate(items):
    print(f'  {k}. {item["sidoName"]} {item["sggName"]} {item["umdName"]}')
    print(f'     TM(X, Y): {item["tmX"]}, {item["tmY"]}')

In [ ]:
# 대전 지역 선택
selected = items[0]
TM_X = selected['tmX']
TM_Y = selected['tmY']
print(f'선택: {selected["sidoName"]} {selected["sggName"]} → TM({TM_X}, {TM_Y})')

In [ ]:
req_url = 'http://apis.data.go.kr/B552584/MsrstnInfoInqireSvc/getNearbyMsrstnList'

req_parameter = {
    'serviceKey': API_KEY, 'tmX': TM_X, 'tmY': TM_Y,
    'ver': 1.1, 'returnType': output_type
}
dict_data2 = requests.get(req_url, params=req_parameter).json()

total2   = dict_data2['response']['body']['totalCount']
stations = dict_data2['response']['body']['items']

print(f'[근접 측정소] {total2}개')
for s in stations:
    print(f'  - {s["stationName"]}  ({s["tm"]} km)')
    print(f'    {s["addr"]}')